<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/3_wb_qwin_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs: Working on colab run time > 2026.04

In [ ]:
# 1. Remove conflicting packages
!pip uninstall -y torch torchvision torchaudio torchtext \
  triton bitsandbytes \
  transformers tokenizers accelerate peft trl datasets \
  numpy pandas pyarrow

In [ ]:
# 2. Install PyTorch CUDA 11.8 stack
# Moving from torch 2.2.0 -> 2.3.1 is safer on modern Colab / Python 3.12.
# PyTorch publishes a CUDA 11.8 wheel for torch 2.3.1.
!pip install --no-cache-dir \
  torch==2.3.1 \
  torchvision==0.18.1 \
  torchaudio==2.3.1 \
  --index-url https://download.pytorch.org/whl/cu118

# <font color = red> RESTART RUNTIME HERE and then run below

In [ ]:
# 3. Install Hugging Face + data stack
# bitsandbytes 0.45.4 avoids the old triton.ops import issue better than 0.43.x.
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  pyarrow==15.0.2 \
  transformers==4.38.2 \
  tokenizers==0.15.2 \
  peft==0.9.0 \
  accelerate==0.27.2 \
  trl==0.8.1 \
  datasets==2.18.0 \
  bitsandbytes==0.45.4 \
  scipy==1.12.0 \
  fsspec==2024.2.0

# <font color = red> RESTART RUNTIME HERE and then run below

# Imports

In [ ]:
import torch
import tokenizers
import peft
import accelerate
import trl
from datasets import Dataset
import bitsandbytes as bnb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
#import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm
from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer
)
from sklearn.model_selection import train_test_split
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import SFTTrainer
import gc
import wandb
import os

## Weights & Biases (W&B) and Project Setup

We will integrate Weights & Biases (W&B) to track experiments, log metrics, and save model artifacts. The project structure will be organized as follows:

`{base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/`

Each W&B experiment will save its final checkpoints and other necessary artifacts within this structure.

## Fine-tuning Functions

Now, let's refactor the fine-tuning process into modular functions to improve readability, reusability, and integration with W&B.

In [ ]:
def load_and_prepare_data(data_file_path, tokenizer):
    """Loads data from a parquet file, formats it using a chat template, and returns Hugging Face Datasets for training and evaluation."""
    print(f"Loading data from {data_file_path}...")
    full_df = pd.read_parquet(data_file_path)

    # Define the formatting function for the chat template
    def format_chat_template(row):
        product_types = [
            'Checking or savings account',
            'Money transfer, virtual currency, or money service',
            'Vehicle loan or lease',
            'Credit card',
            'Payday loan, title loan, personal loan, or advance loan',
            'Credit reporting or other personal consumer reports',
            'Mortgage',
            'Student loan',
            'Debt collection',
            'Debt or credit management'
        ]
        messages = [
            {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
            {"role": "user", "content": f"Complaint: {row['input_data']}\nPredefined Product Categories: {product_types}"},
            {"role": "assistant", "content": row['ground_truth_data']}
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

    # Convert DataFrames to Hugging Face Datasets
    train_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'train']).map(format_chat_template, remove_columns=list(full_df.columns))
    eval_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'validation']).map(format_chat_template, remove_columns=list(full_df.columns))
    test_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'test']).map(format_chat_template, remove_columns=list(full_df.columns))

    print("Training data sample:")
    print(train_dataset[0]['text'])
    print("\nEvaluation data sample:")
    print(eval_dataset[0]['text'])

    return train_dataset, eval_dataset, test_dataset

In [ ]:
def load_qwen_model(model_name, model_kwargs):
    """Loads the Qwen model with 4-bit quantization configuration."""
    print(f"Loading Qwen model: {model_name} with 4-bit quantization...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=model_kwargs.get("load_in_4bit", True),
        bnb_4bit_quant_type=model_kwargs.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=model_kwargs.get("bnb_4bit_compute_dtype", torch.bfloat16),
        bnb_4bit_use_double_quant=model_kwargs.get("bnb_4bit_use_double_quant", False),
        llm_int8_skip_modules=model_kwargs.get("llm_int8_skip_modules", None), # Add this for specific layers to skip quantization
        # Add offload_dir for models too large for GPU
        offload_dir=model_kwargs.get("offload_dir", None),
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1 # Recommended for Qwen
    return model

In [ ]:
def setup_lora_config(model, lora_kwargs):
    """Prepares the model for k-bit training and applies LoRA configuration."""
    print("Setting up LoRA configuration...")
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=lora_kwargs.get("lora_r", 16),
        lora_alpha=lora_kwargs.get("lora_alpha", 32),
        target_modules=lora_kwargs.get("lora_target_modules", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
        lora_dropout=lora_kwargs.get("lora_dropout", 0.05),
        bias=lora_kwargs.get("lora_bias", "none"),
        task_type=lora_kwargs.get("lora_task_type", "CAUSAL_LM"),
    )

    model = get_peft_model(model, lora_config)
    print(model.print_trainable_parameters())
    return model, lora_config

In [ ]:
def configure_wandb_training_args(output_dir, run_name, training_kwargs):
    """Configures TrainingArguments for SFTTrainer with W&B integration."""
    print(f"Configuring training arguments for W&B run: {run_name}...")
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=training_kwargs.get("num_train_epochs", 3),
        per_device_train_batch_size=training_kwargs.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=training_kwargs.get("gradient_accumulation_steps", 4),
        gradient_checkpointing=training_kwargs.get("gradient_checkpointing", True),
        optim=training_kwargs.get("optim", "paged_adamw_8bit"),
        logging_steps=training_kwargs.get("logging_steps", 10),
        learning_rate=training_kwargs.get("learning_rate", 2e-4),
        fp16=training_kwargs.get("fp16", True),
        bf16=training_kwargs.get("bf16", False),
        tf32=training_kwargs.get("tf32", False),
        max_grad_norm=training_kwargs.get("max_grad_norm", 0.3),
        warmup_ratio=training_kwargs.get("warmup_ratio", 0.03),
        lr_scheduler_type=training_kwargs.get("lr_scheduler_type", "constant"),
        report_to="wandb",
        run_name=run_name,
        evaluation_strategy=training_kwargs.get("evaluation_strategy", "epoch"),
        save_strategy=training_kwargs.get("save_strategy", "epoch"),
        load_best_model_at_end=training_kwargs.get("load_best_model_at_end", True),
        metric_for_best_model=training_kwargs.get("metric_for_best_model", "eval_loss"),
        greater_is_better=training_kwargs.get("greater_is_better", False),
    )
    return training_args

In [ ]:
def run_fine_tuning(model, train_dataset, eval_dataset, lora_config, tokenizer, training_args, sft_trainer_kwargs):
    """Initializes and runs the SFTTrainer."""
    print("\nStarting Qwen fine-tuning...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
        args=training_args,
        max_seq_length=sft_trainer_kwargs.get("max_seq_length", 512), # Maximum sequence length for inputs
        dataset_text_field="text", # Field in dataset containing the formatted text
        packing=sft_trainer_kwargs.get("packing", True), # Pack multiple examples into one sequence for efficiency
    )

    trainer.train()
    print("\nFine-tuning complete!")

    # Save the adapter in the specified output directory
    adapter_output_dir = os.path.join(training_args.output_dir, "final_adapter")
    trainer.save_model(adapter_output_dir)
    print(f"Final adapter saved to: {adapter_output_dir}")

    return trainer

In [ ]:
def evaluate_finetuned_model(model_name, adapter_path, tokenizer, split_data_path, product_types, model_kwargs):
    """Loads the base model, merges with adapter, and evaluates classification performance."""
    print("\nEvaluating fine-tuned model...")
    # Clear VRAM and local memory before reloading model for evaluation
    gc.collect()
    torch.cuda.empty_cache()

    print("Loading base model and merging adapters... this may take a moment.")

    # Re-create the BitsAndBytesConfig for the base model during evaluation
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=model_kwargs.get("load_in_4bit", True),
        bnb_4bit_quant_type=model_kwargs.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=model_kwargs.get("bnb_4bit_compute_dtype", torch.bfloat16),
        bnb_4bit_use_double_quant=model_kwargs.get("bnb_4bit_use_double_quant", False),
        llm_int8_skip_modules=model_kwargs.get("llm_int8_skip_modules", None),
        offload_dir=model_kwargs.get("offload_dir", None), # Ensure offload_dir is passed here
    )

    # Reload the base model with 4-bit quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    # Load the PEFT model and merge, passing offload_dir for accelerate's dispatch_model
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        adapter_path
    )

    print("Merging adapters...")
    finetuned_model = finetuned_model.merge_and_unload()
    finetuned_model.eval()

    print("Fine-tuned model loaded and adapters merged successfully.")

    # Load the full data from the provided path and filter for the test set
    print(f"Loading original test data from {split_data_path} for evaluation...")
    full_df_from_path = pd.read_parquet(split_data_path)
    test_df_original = full_df_from_path[full_df_from_path['tts_label'] == 'test']

    # Update the classification function to use the fine-tuned model
    def classify_product_with_finetuned_qwen(
        complaint_row,
        model,
        tokenizer,
        product_types
        ):
        messages = [
            {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
            {"role": "user", "content": f"Complaint: {complaint_row['input_data']}\nPredefined Product Categories: {product_types}"}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        try:
            generated_ids = model.generate(
                model_inputs.input_ids,
                max_new_tokens=50,
                do_sample=False,
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id # Explicitly use EOS token for stopping
            )
            input_len = model_inputs.input_ids.shape[1]
            response_text = tokenizer.decode(generated_ids[0, input_len:], skip_special_tokens=True).strip()

            # --- Fix for garbled output: Clean the response text before matching ---
            # Remove the known repetitive garbled Chinese characters
            cleaned_response_text = response_text.replace("荣荣荣荣", "").strip()

            predicted_category = "Unknown"
            for p_type in product_types:
                # Check for direct containment, case-insensitive, in the cleaned text
                if p_type.lower() in cleaned_response_text.lower():
                    predicted_category = p_type
                    break
            # The problematic fallback was removed in a previous turn, no need to add anything here.

            return predicted_category
        except Exception as e:
            print(f"Error classifying complaint: {e}")
            return "Error_Classification"

    print("Running fine-tuned Qwen classification on the evaluation subset (first 100 rows)...")
    # Take first 100 rows of the evaluation DataFrame (assuming eval_df contains the original data)
    eval_subset_df = test_df_original.head(100).copy() # Use the internally loaded test_df

    # Apply the fine-tuned Qwen classification function with a progress bar
    tqdm.pandas() # Activate tqdm for pandas apply functions

    finetuned_qwen_inf_results = eval_subset_df.progress_apply( # Changed .apply to .progress_apply
        lambda row: classify_product_with_finetuned_qwen(row, finetuned_model, tokenizer, product_types),
        axis=1
    )

    eval_subset_df['finetuned_qwen_classified_product'] = finetuned_qwen_inf_results

    print("\nFine-tuned Qwen Classified Product Value Counts:")
    display(eval_subset_df['finetuned_qwen_classified_product'].value_counts())

    print("\nComparison of Original vs. Fine-tuned Qwen Classified Products (first 10 rows):")
    display(eval_subset_df[['ground_truth_data', 'finetuned_qwen_classified_product', 'input_data']].head(10))

    correct_finetuned_qwen_classifications = (eval_subset_df['ground_truth_data'] == eval_subset_df['finetuned_qwen_classified_product']).sum()
    total_finetuned_qwen_classifications = len(eval_subset_df)
    accuracy_finetuned_qwen = correct_finetuned_qwen_classifications / total_finetuned_qwen_classifications
    print(f"\nFine-tuned Qwen Classification Accuracy (vs. original 'ground_truth_data' column): {accuracy_finetuned_qwen:.2f}")

    return accuracy_finetuned_qwen, eval_subset_df

# Main Execution Section

In [ ]:
# USER INPUTS:
drive_dir_name = "CFG_complaints"
project_name = "product_classification"
project_num = 4

# Define the model name globally
qwen_model_name = "Qwen/Qwen1.5-4B-Chat"

# Load tokenizer globally and configure padding
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token # Qwen uses EOS as padding token by default
tokenizer.padding_side = "right" # Or "left" depending on preference for inference

# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

### Set wandb names
wb_project_name = f"{project_name}_pjnum_{project_num}"

### Build directory paths

# Base directory path
base_dir = "/content/drive/MyDrive/"

# Build path for base project directory
#     {base_dir}/{drive_dir_name}/projects/{project_name}
base_project_dir = os.path.join(
    base_dir,
    drive_dir_name,
    "projects",
    project_name
)

# Build path for split data path
#     {base_dir}/{drive_dir_name}/projects/{project_name}/split_data/split_data.parquet
split_data_path = os.path.join(
    base_project_dir,
    f"project_{project_num}",
    "split_data",
    "split_data.parquet"
)

# Build base path for experiments
#    {base_dir}/{drive_dir_name}/projects/{project_name}/experiments/exp_{x}
exp_base_path = os.path.join(
    base_project_dir,
    f"project_{project_num}",
    "experiments",
)

# Configure for Each New Experiment

In [ ]:
experiment_config = {
    # Model Kwargs for load_qwen_model
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.bfloat16,
    "bnb_4bit_use_double_quant": False,
    "llm_int8_skip_modules": None, # Specify module names to skip 8-bit quantization, e.g., ["lm_head"]
    "offload_dir": "/content/offload", # Directory for offloading model layers to disk if GPU memory runs out

    # LoRA Kwargs for setup_lora_config (prefixed with 'lora_')
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_dropout": 0.05,
    "lora_bias": "none",
    "lora_task_type": "CAUSAL_LM",

    # Training Arguments for configure_wandb_training_args
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
    "logging_steps": 10,
    "learning_rate": 2e-4,
    "fp16": True,
    "bf16": False, # Set to True if your GPU supports bfloat16
    "tf32": False,
    "max_grad_norm": 0.3,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "constant",
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,

    # SFTTrainer Kwargs for run_fine_tuning
    "max_seq_length": 512,
    "packing": True,
}

# Run for each new experiment

In [ ]:
# Main execution block for fine-tuning

# 0. Create new experiment directory
# Find existing experiment runs and determine the next experiment number
try:
  existing_experiments = [
      d for d in os.listdir(exp_base_path)
      if os.path.isdir(os.path.join(exp_base_path, d)) and
      d.startswith('exp_')
  ]
except FileNotFoundError:
  os.makedirs(exp_base_path, exist_ok=True)
  existing_experiments = []

if existing_experiments:
    # Extract numbers from experiment folder names (e.g., 'exp_1' -> 1)
    exp_numbers = [
        int(d.split('_')[1]) for d in existing_experiments
        if d.split('_')[1].isdigit()
    ]
    if exp_numbers:
        next_exp_number = max(exp_numbers) + 1
    else:
        next_exp_number = 1 # No valid exp_N folders found, start with 1
else:
    next_exp_number = 1

# Construct the new experiment directory path
current_experiment_dir = os.path.join(exp_base_path, f"exp_{next_exp_number}")
os.makedirs(current_experiment_dir, exist_ok=True)

print(f"New experiment directory created: {current_experiment_dir}")

# 1. Initialize W&B
wb_run_name = f"project_{project_num}_experiment_{next_exp_number}"

# Use a copy of the config dictionary for wandb.init to avoid modifying the original
wandb_config = experiment_config.copy()
wandb_config["model_name"] = qwen_model_name # Add model_name to wandb config

wandb.init(
    project=wb_project_name,
    name=wb_run_name,
    config=wandb_config
)


# 2. Load and prepare data (Hugging Face Datasets format)
train_dataset, eval_dataset, test_dataset_for_eval = load_and_prepare_data(
    split_data_path,
    tokenizer
)

# Extract model kwargs
model_kwargs = {k: v for k, v in experiment_config.items() if k in ["load_in_4bit", "bnb_4bit_quant_type", "bnb_4bit_compute_dtype", "bnb_4bit_use_double_quant", "llm_int8_skip_modules", "offload_dir"]}
# 3. Load Qwen model
model = load_qwen_model(qwen_model_name, model_kwargs)

# Extract lora kwargs
lora_kwargs = {k: v for k, v in experiment_config.items() if k.startswith("lora_")}
# 4. Setup LoRA
model, lora_config = setup_lora_config(model, lora_kwargs)

# 5. Configure Training Arguments
# Training output directory will be inside the project structure
training_output_dir = os.path.join(current_experiment_dir, "training_output")
os.makedirs(training_output_dir, exist_ok=True)

# Extract training kwargs
training_kwargs = {k: v for k, v in experiment_config.items() if k in [
    "num_train_epochs", "per_device_train_batch_size", "gradient_accumulation_steps",
    "gradient_checkpointing", "optim", "logging_steps", "learning_rate", "fp16",
    "bf16", "tf32", "max_grad_norm", "warmup_ratio", "lr_scheduler_type",
    "evaluation_strategy", "save_strategy", "load_best_model_at_end",
    "metric_for_best_model", "greater_is_better"
]}
training_args = configure_wandb_training_args(
    training_output_dir,
    wb_run_name,
    training_kwargs
)

# Extract SFTTrainer kwargs
sft_trainer_kwargs = {k: v for k, v in experiment_config.items() if k in ["max_seq_length", "packing"]}
# 6. Run Fine-tuning
trainer = run_fine_tuning(
    model,
    train_dataset,
    eval_dataset,
    lora_config,
    tokenizer,
    training_args,
    sft_trainer_kwargs
)

# 7. Evaluate the fine-tuned model
final_adapter_path = os.path.join(training_output_dir, "final_adapter")
product_types = [
    'Checking or savings account',
    'Money transfer, virtual currency, or money service',
    'Vehicle loan or lease',
    'Credit card',
    'Payday loan, title loan, personal loan, or advance loan',
    'Credit reporting or other personal consumer reports',
    'Mortgage',
    'Student loan',
    'Debt collection',
    'Debt or credit management'
]

accuracy, eval_results_df = evaluate_finetuned_model(
    qwen_model_name,
    final_adapter_path,
    tokenizer,
    test_dataset_for_eval,
    product_types,
    model_kwargs # Pass model_kwargs for offload_dir
  )

# Log final accuracy to W&B
wandb.log({"final_evaluation_accuracy": accuracy})

# Save the evaluation results DataFrame as a W&B artifact
artifact = wandb.Artifact("evaluation_results", type="dataset")
with artifact.new_file("eval_results.csv", "w") as f:
    eval_results_df.to_csv(f, index=False)
wandb.log_artifact(artifact)

print("\nFine-tuning and evaluation complete. Check W&B for experiment details.")

# 8. Finish W&B run
wandb.finish()

In [ ]:
model_kwargs = {k: v for k, v in experiment_config.items() if k in ["load_in_4bit", "bnb_4bit_quant_type", "bnb_4bit_compute_dtype", "bnb_4bit_use_double_quant", "llm_int8_skip_modules", "offload_dir"]}

accuracy, eval_results_df = evaluate_finetuned_model(
    qwen_model_name,
    final_adapter_path,
    tokenizer,
    split_data_path, # Now correctly passed as a positional argument
    product_types,
    model_kwargs
  )

In [ ]:
eval_results_df.sample(10)[['ground_truth_data', 'finetuned_qwen_classified_product']]

In [ ]:
final_adapter_path

In [ ]:
# 1. Clear VRAM and local memory
if 'model' in locals(): del model
if 'trainer' in locals(): del trainer
gc.collect()
torch.cuda.empty_cache()

# Fixing Evaluation

In [ ]:
offload_dir = "./offload_temp"

In [ ]:
adaptor_path = '/content/drive/MyDrive/CFG_complaints/projects/product_classification/project_4/experiments/exp_1/training_output/final_adapter'

In [ ]:

print("Loading base model and merging adapters... this may take a moment.")

# 2. Reload the base model with offloading enabled
base_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder=offload_dir,
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Load the PEFT model with offload_folder and merge
# Passing offload_folder here fixes the ValueError during dispatch
finetuned_model = PeftModel.from_pretrained(
    base_model,
    adaptor_path,
    offload_folder=offload_dir
)

print("Merging adapters...")
finetuned_model = finetuned_model.merge_and_unload()
finetuned_model.eval()

print("Fine-tuned model loaded and adapters merged successfully.")

In [ ]:
eval_df = pd.read_parquet(split_data_path)
eval_df = eval_df[eval_df['tts_label'] == 'test']

In [ ]:

# Update the classification function to use the fine-tuned model
def classify_product_with_finetuned_qwen(
    complaint_row,
    model,
    tokenizer,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    messages = [
        {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
        {"role": "user", "content": f"Complaint: {complaint_row['input_data']}\nPredefined Product Categories: {product_types}"}
    ]

    # Tokenize the prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    try:
        # Generate response from Qwen model
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=50, # Limit the output length
            do_sample=False, # Use greedy decoding for deterministic classification
            temperature=0.0, # Set temperature to 0 for deterministic output
        )
        # Decode only the newly generated tokens
        input_len = model_inputs.input_ids.shape[1]
        response_text = tokenizer.decode(generated_ids[0, input_len:], skip_special_tokens=True).strip()

        # Post-process the response to get only the product category
        predicted_category = "Unknown"
        for p_type in product_types:
            if p_type.lower() in response_text.lower():
                predicted_category = p_type
                break
        # Fallback to the direct response if no predefined category is found and response is short
        if predicted_category == "Unknown" and len(response_text.split()) < 5: # Assuming category is short
             predicted_category = response_text # Take the model's direct output

        return predicted_category
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

print("Running fine-tuned Qwen classification on the evaluation subset (first 10 rows)...")
# Apply the fine-tuned Qwen classification function to the evaluation set
finetuned_qwen_inf_df = eval_df.apply(
    lambda row: classify_product_with_finetuned_qwen(row, finetuned_model, tokenizer),
    axis=1
)

# Add the Qwen classifications to a new column in eval_df for comparison
eval_df['finetuned_qwen_classified_product'] = finetuned_qwen_inf_df


In [ ]:
eval_df['pred_true'] = eval_df.ground_truth_data == eval_df.finetuned_qwen_classified_product
eval_df.pred_true.value_counts()

In [ ]:
eval_df.groupby('ground_truth_data').pred_true.value_counts(dropna = False).to_frame().reset_index()

In [ ]:

# Display the value counts of the fine-tuned Qwen-classified products
print("\nFine-tuned Qwen Classified Product Value Counts:")
display(eval_df['finetuned_qwen_classified_product'].value_counts())

# Display a comparison of original vs. fine-tuned Qwen classified for a few rows
print("\nComparison of Original vs. Fine-tuned Qwen Classified Products (first 10 rows):")
display(eval_df[['product', 'finetuned_qwen_classified_product', 'complaint_what_happened']])

# Calculate accuracy on the evaluation subset
correct_finetuned_qwen_classifications = (eval_df['product'] == eval_df['finetuned_qwen_classified_product']).sum()
total_finetuned_qwen_classifications = len(eval_df)
accuracy_finetuned_qwen = correct_finetuned_qwen_classifications / total_finetuned_qwen_classifications
print(f"\nFine-tuned Qwen Classification Accuracy (vs. original 'product' column): {accuracy_finetuned_qwen:.2f}")

In [ ]:
eval_df

## TODO:
• Make prompts one of the things we can test